# Análisis futbolístico de las jugadas por equipo

Este cuaderno estudia las **jugadas elegibles** del corpus local (al menos tres pases completados del equipo protagonista). Revisa los 24 clusters de SVD + MiniBatchKMeans y les asigna una **familia sugerida** y una **jugada aproximada** a partir de sus n-gramas y ejemplos. El análisis principal usa esos nombres futbolísticos: frecuencia, longitud, pases, diferencias entre equipos y temporadas. Los IDs técnicos quedan para auditoría.

**Estado de la evidencia:** el piloto rechazó esta clasificación por estabilidad insuficiente entre semillas. Las descripciones son hipótesis exploratorias, **no clases tácticas validadas**. `Sin patrón claro` se muestra por separado; su cluster más cercano no se cuenta como tipo de jugada.

Ejecutar desde la raíz del repositorio con el kernel de `.venv`. El notebook lee artefactos locales, no reentrena modelos ni modifica los datos de origen. Necesita `pandas`, `numpy`, `matplotlib` y `IPython`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

RAIZ = Path.cwd().resolve()
RUTA_CLASIFICACION = RAIZ / "datos_clasificacion_svd_kmeans_completo_exploratorio"
RUTA_CORPUS = RAIZ / "datos_preparados_corpus"
RUTA_ETIQUETAS = RUTA_CLASIFICACION / "clasificacion_exploratoria.jsonl"
RUTA_EVALUACION = RUTA_CLASIFICACION / "evaluacion_completa.json"

for ruta in (RUTA_ETIQUETAS, RUTA_EVALUACION, RUTA_CORPUS):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecute el cuaderno desde la raíz del proyecto y genere primero los artefactos.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.22})
evaluacion = json.loads(RUTA_EVALUACION.read_text(encoding="utf-8"))
assert evaluacion["decision"] == "rechazado"
display(pd.DataFrame([{
    "categoría": categoria,
    "clusters": datos["k"],
    "silueta_coseno_muestra": datos["silueta_coseno_muestra_2000"],
    "decisión": evaluacion["decision"],
} for categoria, datos in evaluacion["categorias"].items()]))

## 1. Revisión de las clasificaciones

Las fichas contienen n-gramas de acciones (secuencias de dos o tres eventos), ejemplos y el contexto de reanudación. Una misma ficha puede reunir saques de banda, faltas y córners. Por eso las etiquetas siguientes describen la **forma dominante de desarrollar el balón** y no identifican una jugada ensayada concreta. Los códigos `c01` a `c16` son IDs de cluster y se mantienen junto a cada descripción.

La dirección de pase se refiere al avance del balón en el sistema de coordenadas normalizado; no demuestra por sí sola intención táctica, zona del campo o éxito de la jugada. La presión es un evento rival presente en los n-gramas, no una medida de intensidad defensiva.

In [ ]:
# Interpretación manual de las fichas del modelo completo. Cambiar estos nombres no cambia la clasificación.
INTERPRETACIONES = {
    "balon_parado_k08_c01": ("Progresión", "Reanudación con pase de avance medio y apoyo corto"),
    "balon_parado_k08_c02": ("Bajo presión", "Reanudación con pase de avance medio tras presión"),
    "balon_parado_k08_c03": ("Circulación", "Reanudación con circulación lateral corta"),
    "balon_parado_k08_c04": ("Reciclaje", "Reanudación con pase largo hacia atrás y nueva circulación"),
    "balon_parado_k08_c05": ("Progresión", "Reanudación con avances cortos y apoyos"),
    "balon_parado_k08_c06": ("Reciclaje", "Reanudación con pase atrás medio y circulación"),
    "balon_parado_k08_c07": ("Mixto", "Reanudación con avance y recuperación; perfil heterogéneo"),
    "balon_parado_k08_c08": ("Circulación", "Reanudación con pase lateral medio y apoyos"),
    "juego_abierto_k16_c01": ("Circulación", "Conducción de avance seguida de pase lateral"),
    "juego_abierto_k16_c02": ("Progresión", "Pase de avance medio seguido de conducción de avance"),
    "juego_abierto_k16_c03": ("Bajo presión", "Circulación lateral media tras presión"),
    "juego_abierto_k16_c04": ("Bajo presión", "Regate y conducción tras presión"),
    "juego_abierto_k16_c05": ("Progresión", "Circulación lateral seguida de avance corto"),
    "juego_abierto_k16_c06": ("Progresión", "Pase largo de avance y conducción de avance"),
    "juego_abierto_k16_c07": ("Progresión", "Pase de avance medio y conducción corta de avance"),
    "juego_abierto_k16_c08": ("Bajo presión", "Pase de avance medio tras presión"),
    "juego_abierto_k16_c09": ("Circulación", "Circulación lateral media prolongada"),
    "juego_abierto_k16_c10": ("Progresión", "Progresión con pases medios y apoyos laterales"),
    "juego_abierto_k16_c11": ("Progresión", "Pase largo de avance con apoyo posterior"),
    "juego_abierto_k16_c12": ("Circulación", "Circulación lateral corta y apoyos"),
    "juego_abierto_k16_c13": ("Bajo presión", "Pase lateral corto tras presión"),
    "juego_abierto_k16_c14": ("Bajo presión", "Conducción lateral después de presión"),
    "juego_abierto_k16_c15": ("Reciclaje", "Pase atrás medio tras presión"),
    "juego_abierto_k16_c16": ("Circulación", "Pase lateral largo y apoyos; posible cambio de orientación"),
}

fichas = [ficha for datos in evaluacion["categorias"].values() for ficha in datos["fichas"]]
assert set(INTERPRETACIONES) == {f["cluster"] for f in fichas}
df_catalogo = pd.DataFrame([{
    "cluster": f["cluster"],
    "categoria": f["cluster"].split("_k")[0],
    "familia_sugerida": INTERPRETACIONES[f["cluster"]][0],
    "jugada_aproximada": INTERPRETACIONES[f["cluster"]][1],
    "ngramas_principales": " | ".join(n["ngramas"] for n in f["ngramas"][:3]),
    "reanudaciones_evaluacion": f["reanudaciones_evaluacion"],
    "ejemplo_play_id": f["ejemplos"][0]["play_id"],
    "eventos_medios_evaluacion": f["longitud_media_eventos_evaluacion"],
} for f in fichas]).sort_values(["categoria", "cluster"]).reset_index(drop=True)
display(df_catalogo)
display(Markdown("**Lectura:** estas familias sirven para resumir patrones observados. Clusters de una misma familia siguen siendo grupos distintos y ninguna familia equivale automáticamente a contraataque, centro, transición o estrategia ensayada."))

### Ejemplos auditables

La siguiente función permite ver la secuencia inicial de eventos de una jugada representativa. El catálogo muestra su `play_id`; la ficha registra la ruta. Revisar ejemplos es especialmente útil para los clusters marcados como mixtos o bajo presión.

In [ ]:
def mostrar_ejemplo(cluster, max_eventos=18):
    ficha = next(f for f in fichas if f["cluster"] == cluster)
    ejemplo = ficha["ejemplos"][0]
    ruta = RUTA_CORPUS / Path(ejemplo["manifiesto"]).parent / ejemplo["archivo"]
    jugada = json.loads(ruta.read_text(encoding="utf-8"))
    columnas = jugada["columns"]
    df_eventos = pd.DataFrame(jugada["data"], columns=columnas)
    def nombre(valor):
        return valor.get("name") if isinstance(valor, dict) else valor
    salida = df_eventos[["index", "type", "team", "play_pattern"]].head(max_eventos).copy()
    for col in ("type", "team", "play_pattern"):
        salida[col] = salida[col].map(nombre)
    print(f"{cluster} | {ejemplo['play_id']} | {ruta.relative_to(RAIZ)}")
    display(salida)

mostrar_ejemplo("juego_abierto_k16_c04")

## 2. Construcción de los dataframes

Se enlaza cada etiqueta exploratoria con el `equipo_protagonista` y la temporada de su manifiesto. Se comprueba que los IDs sean únicos, que el cruce cubra todas las filas y que la categoría coincida. `df_jugadas` conserva la familia y la jugada aproximada como columnas consultables. El archivo de clasificación cubre el corpus completo, no solo el piloto.

In [ ]:
df_etiquetas = pd.read_json(RUTA_ETIQUETAS, lines=True)
assert df_etiquetas["play_id"].is_unique, "Hay IDs duplicados en la clasificación"
assert len(df_etiquetas) == evaluacion["total_jugadas"], "El total no coincide con la evaluación"

metadatos = []
for ruta_relativa in df_etiquetas["manifiesto"].unique():
    manifiesto = json.loads((RUTA_CORPUS / ruta_relativa).read_text(encoding="utf-8"))
    for jugada in manifiesto["jugadas"]:
        metadatos.append({
            "play_id": jugada["play_id"],
            "equipo": jugada["equipo_protagonista"],
            "temporada": manifiesto["temporada_nombre"],
            "categoria_manifiesto": jugada["categoria_clasificacion"],
        })

df_metadatos = pd.DataFrame(metadatos)
assert df_metadatos["play_id"].is_unique, "Hay IDs duplicados en los manifiestos"
df_jugadas = df_etiquetas.merge(df_metadatos, on="play_id", how="left", validate="one_to_one", indicator=True)
assert (df_jugadas["_merge"] == "both").all(), "Hay etiquetas sin manifiesto"
assert (df_jugadas["categoria"] == df_jugadas["categoria_manifiesto"]).all(), "Categorías inconsistentes"
df_jugadas = df_jugadas.drop(columns=["_merge", "categoria_manifiesto"])
df_jugadas = df_jugadas.rename(columns={"etiqueta_exploratoria": "cluster_analisis"})
df_jugadas = df_jugadas.merge(
    df_catalogo[["cluster", "familia_sugerida", "jugada_aproximada"]],
    left_on="cluster_analisis", right_on="cluster", how="left", validate="many_to_one"
).drop(columns="cluster")
df_jugadas["familia_sugerida"] = df_jugadas["familia_sugerida"].fillna("Sin patrón claro")
df_jugadas["jugada_aproximada"] = df_jugadas["jugada_aproximada"].fillna("Sin patrón claro")
assert df_jugadas["equipo"].notna().all()
assert df_jugadas.loc[df_jugadas["cluster_analisis"] != "Sin patrón claro", "familia_sugerida"].ne("Sin patrón claro").all()
print(f"{len(df_jugadas):,} jugadas | {df_jugadas['equipo'].nunique()} equipos | {df_jugadas['partido_id'].nunique()} partidos")
display(df_jugadas[["play_id", "equipo", "temporada", "categoria", "familia_sugerida", "jugada_aproximada", "pases_completados", "cluster_analisis"]].head())

In [ ]:
# Resúmenes de jugadas y familias: una fila por interpretación futbolística y fase.
totales_fase = df_jugadas.groupby("categoria").size().rename("jugadas_corpus_fase")
df_estadisticas_jugadas = (
    df_jugadas.groupby(["categoria", "familia_sugerida", "jugada_aproximada"], dropna=False)
    .agg(jugadas=("play_id", "size"), equipos=("equipo", "nunique"),
         partidos=("partido_id", "nunique"), pases_medios=("pases_completados", "mean"),
         pases_medianos=("pases_completados", "median"),
         eventos_medios=("total_eventos", "mean"),
         eventos_medianos=("total_eventos", "median"),
         distancia_mediana=("distancia_centroide", "median"),
         clusters_origen=("cluster_analisis", lambda s: ", ".join(sorted(s.unique()))))
    .join(totales_fase, on="categoria").reset_index()
)
df_estadisticas_jugadas["porcentaje_fase"] = 100 * df_estadisticas_jugadas["jugadas"] / df_estadisticas_jugadas["jugadas_corpus_fase"]
df_estadisticas_jugadas = df_estadisticas_jugadas.sort_values(["categoria", "jugadas"], ascending=[True, False]).reset_index(drop=True)

df_estadisticas_familias = (
    df_jugadas.groupby(["categoria", "familia_sugerida"], dropna=False)
    .agg(jugadas=("play_id", "size"), equipos=("equipo", "nunique"),
         partidos=("partido_id", "nunique"), pases_medios=("pases_completados", "mean"),
         eventos_medios=("total_eventos", "mean"))
    .join(totales_fase, on="categoria").reset_index()
)
df_estadisticas_familias["porcentaje_fase"] = 100 * df_estadisticas_familias["jugadas"] / df_estadisticas_familias["jugadas_corpus_fase"]
df_estadisticas_familias = df_estadisticas_familias.sort_values(["categoria", "jugadas"], ascending=[True, False]).reset_index(drop=True)

# Resúmenes por equipo. "porcentaje_fase" siempre usa todas las jugadas elegibles
# del equipo en esa fase, incluido "Sin patrón claro".
por_equipo = df_jugadas.groupby("equipo").agg(
    jugadas_equipo=("play_id", "size"), partidos_equipo=("partido_id", "nunique")).reset_index()
por_fase = df_jugadas.groupby(["equipo", "categoria"]).size().rename("jugadas_fase").reset_index()

df_estadisticas_equipos = (
    df_jugadas.groupby(["equipo", "categoria", "familia_sugerida", "jugada_aproximada"], dropna=False)
    .agg(jugadas=("play_id", "size"), partidos_con_jugada=("partido_id", "nunique"),
         pases_medios=("pases_completados", "mean"), eventos_medios=("total_eventos", "mean"))
    .reset_index()
    .merge(por_fase, on=["equipo", "categoria"], validate="many_to_one")
    .merge(por_equipo, on="equipo", validate="many_to_one")
    .merge(df_estadisticas_jugadas[["categoria", "familia_sugerida", "jugada_aproximada", "porcentaje_fase"]]
           .rename(columns={"porcentaje_fase": "porcentaje_fase_corpus"}),
           on=["categoria", "familia_sugerida", "jugada_aproximada"], validate="many_to_one")
)
df_estadisticas_equipos["porcentaje_fase"] = 100 * df_estadisticas_equipos["jugadas"] / df_estadisticas_equipos["jugadas_fase"]
df_estadisticas_equipos["porcentaje_equipo"] = 100 * df_estadisticas_equipos["jugadas"] / df_estadisticas_equipos["jugadas_equipo"]
df_estadisticas_equipos["jugadas_por_partido"] = df_estadisticas_equipos["jugadas"] / df_estadisticas_equipos["partidos_equipo"]
df_estadisticas_equipos["indice_uso_relativo"] = df_estadisticas_equipos["porcentaje_fase"] / df_estadisticas_equipos["porcentaje_fase_corpus"]
df_estadisticas_equipos = df_estadisticas_equipos.sort_values(["equipo", "categoria", "jugadas"], ascending=[True, True, False]).reset_index(drop=True)

df_estadisticas_familias_equipos = (
    df_jugadas.groupby(["equipo", "categoria", "familia_sugerida"], dropna=False)
    .agg(jugadas=("play_id", "size"), partidos_con_familia=("partido_id", "nunique"),
         pases_medios=("pases_completados", "mean"), eventos_medios=("total_eventos", "mean"))
    .reset_index()
    .merge(por_fase, on=["equipo", "categoria"], validate="many_to_one")
    .merge(df_estadisticas_familias[["categoria", "familia_sugerida", "porcentaje_fase"]]
           .rename(columns={"porcentaje_fase": "porcentaje_fase_corpus"}),
           on=["categoria", "familia_sugerida"], validate="many_to_one")
)
df_estadisticas_familias_equipos["porcentaje_fase"] = 100 * df_estadisticas_familias_equipos["jugadas"] / df_estadisticas_familias_equipos["jugadas_fase"]
df_estadisticas_familias_equipos["indice_uso_relativo"] = df_estadisticas_familias_equipos["porcentaje_fase"] / df_estadisticas_familias_equipos["porcentaje_fase_corpus"]

df_resumen_equipos = por_equipo.rename(columns={"jugadas_equipo": "jugadas", "partidos_equipo": "partidos"})
df_resumen_equipos["jugadas_por_partido"] = df_resumen_equipos["jugadas"] / df_resumen_equipos["partidos"]
sin_patron = df_jugadas["familia_sugerida"].eq("Sin patrón claro").groupby(df_jugadas["equipo"]).mean()
df_resumen_equipos["porcentaje_sin_patron"] = 100 * df_resumen_equipos["equipo"].map(sin_patron)
for categoria in ("balon_parado", "juego_abierto"):
    conteos = df_jugadas.loc[df_jugadas["categoria"].eq(categoria)].groupby("equipo").size()
    df_resumen_equipos[f"jugadas_{categoria}"] = df_resumen_equipos["equipo"].map(conteos).fillna(0).astype(int)
df_resumen_equipos = df_resumen_equipos.sort_values("jugadas", ascending=False).reset_index(drop=True)

for tabla in (df_estadisticas_jugadas, df_estadisticas_familias,
              df_estadisticas_equipos, df_estadisticas_familias_equipos):
    assert tabla["jugadas"].sum() == len(df_jugadas)
assert df_resumen_equipos["jugadas"].sum() == len(df_jugadas)
display(df_estadisticas_familias)
display(df_estadisticas_jugadas[["categoria", "familia_sugerida", "jugada_aproximada", "jugadas", "porcentaje_fase", "pases_medios", "eventos_medios", "clusters_origen"]])
display(df_estadisticas_equipos.head(20))

## 3. Qué tipos aproximados de jugada aparecen

Las familias resumen varias jugadas aproximadas. Todos los porcentajes de esta sección se calculan **dentro de la fase** (balón parado o juego abierto), con `Sin patrón claro` en el denominador. Los pases y eventos medios describen la longitud observada de la secuencia, no su éxito. Los nombres son interpretaciones del modelo exploratorio.

In [ ]:
resumen_fase = df_jugadas.groupby("categoria").agg(
    jugadas=("play_id", "size"), equipos=("equipo", "nunique"),
    pases_medianos=("pases_completados", "median"), eventos_medianos=("total_eventos", "median")
).reset_index()
resumen_fase["porcentaje_total"] = 100 * resumen_fase["jugadas"] / len(df_jugadas)
display(resumen_fase)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(resumen_fase["categoria"], resumen_fase["jugadas"], color=["#4878A8", "#E3983B"])
ax.set(title="Jugadas elegibles por fase", ylabel="Número de jugadas")
for i, row in resumen_fase.iterrows():
    ax.text(i, row["jugadas"], f"{row['porcentaje_total']:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [ ]:
# Familias: frecuencia y composición dentro de cada fase.
tabla_familias = df_estadisticas_familias.pivot(index="categoria", columns="familia_sugerida", values="porcentaje_fase").fillna(0)
ax = tabla_familias.plot.bar(stacked=True, figsize=(11, 5), colormap="Set2")
ax.set(title="Familias sugeridas de jugadas por fase", ylabel="% de jugadas de la fase", xlabel="Fase")
ax.legend(title="Familia", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Las jugadas aproximadas se muestran por nombre; el cluster de origen está en la tabla anterior.
fig, axs = plt.subplots(1, 2, figsize=(19, 9))
for ax, categoria in zip(axs, ("balon_parado", "juego_abierto")):
    datos = df_estadisticas_jugadas.query("categoria == @categoria and familia_sugerida != 'Sin patrón claro'")
    datos = datos.nlargest(10, "jugadas").sort_values("porcentaje_fase")
    ax.barh(datos["jugada_aproximada"], datos["porcentaje_fase"],
            color="#4878A8" if categoria == "balon_parado" else "#E3983B")
    ax.set(title=categoria.replace("_", " ").title(), xlabel="% de jugadas de la fase")
fig.suptitle("Jugadas aproximadas más frecuentes")
plt.tight_layout()
plt.show()

# Relación entre longitud y frecuencia, por familia y fase.
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
for ax, categoria in zip(axs, ("balon_parado", "juego_abierto")):
    datos = df_estadisticas_familias.query("categoria == @categoria and familia_sugerida != 'Sin patrón claro'")
    ax.scatter(datos["pases_medios"], datos["eventos_medios"], s=datos["jugadas"] / 30,
               alpha=0.55, color="#538773" if categoria == "balon_parado" else "#E3983B")
    for fila in datos.itertuples():
        ax.annotate(fila.familia_sugerida, (fila.pases_medios, fila.eventos_medios), xytext=(5, 4), textcoords="offset points")
    ax.set(title=categoria.replace("_", " ").title(), xlabel="Pases completados medios", ylabel="Eventos medios")
fig.suptitle("Longitud observada de las familias (tamaño = número de jugadas)")
plt.tight_layout()
plt.show()

## 4. Qué jugadas usan los equipos

La matriz muestra el porcentaje de cada familia dentro de las jugadas elegibles de **una misma fase y equipo**. La tabla `df_estadisticas_equipos` permite bajar a la jugada aproximada. `indice_uso_relativo = porcentaje_fase / porcentaje_fase_corpus`: un valor mayor que 1 indica una proporción superior a la del corpus de esa fase, sin probar una preferencia táctica causal. Se exige una muestra mínima para la visualización.

In [ ]:
FASE_MATRIZ = "juego_abierto"  # cambiar a "balon_parado" para comparar reanudaciones
MIN_JUGADAS_FASE = 100
tabla_fase = df_estadisticas_familias_equipos.query("categoria == @FASE_MATRIZ and jugadas_fase >= @MIN_JUGADAS_FASE")
top_equipos = tabla_fase.groupby("equipo")["jugadas_fase"].first().nlargest(15).index
matriz = tabla_fase.pivot(index="equipo", columns="familia_sugerida", values="porcentaje_fase").fillna(0).reindex(top_equipos)
fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(matriz, cmap="YlOrBr", aspect="auto", vmin=0)
ax.set_yticks(range(len(matriz)), matriz.index)
ax.set_xticks(range(len(matriz.columns)), matriz.columns, rotation=35, ha="right")
ax.set(title=f"Familias de jugadas por equipo: {FASE_MATRIZ.replace('_', ' ')}", xlabel="Familia sugerida", ylabel="Equipo")
fig.colorbar(im, ax=ax, label="% de jugadas de la fase del equipo")
plt.tight_layout()
plt.show()

display(df_estadisticas_familias_equipos.query("categoria == @FASE_MATRIZ")
        .sort_values(["equipo", "jugadas"], ascending=[True, False])
        [["equipo", "familia_sugerida", "jugadas", "porcentaje_fase", "indice_uso_relativo"]]
        .head(30))

In [ ]:
# Consulte una jugada por su nombre futbolístico; el ID del cluster queda como referencia en df_catalogo.
JUGADA_OBJETIVO = "Pase largo de avance con apoyo posterior"
MIN_JUGADAS_OBJETIVO = 100
if JUGADA_OBJETIVO not in set(df_estadisticas_jugadas["jugada_aproximada"]):
    raise ValueError("Elija una jugada_aproximada de df_estadisticas_jugadas")
ranking = df_estadisticas_equipos.loc[
    df_estadisticas_equipos["jugada_aproximada"].eq(JUGADA_OBJETIVO)
    & df_estadisticas_equipos["jugadas_fase"].ge(MIN_JUGADAS_OBJETIVO)
].nlargest(12, "porcentaje_fase")
display(ranking[["equipo", "categoria", "familia_sugerida", "jugada_aproximada",
                 "jugadas", "jugadas_fase", "porcentaje_fase", "indice_uso_relativo"]])
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranking["equipo"][::-1], ranking["porcentaje_fase"][::-1], color="#538773")
ax.set(title=f"Equipos que más usan: {JUGADA_OBJETIVO}", xlabel="% de jugadas de la fase")
plt.tight_layout()
plt.show()

## 5. Calidad de asignación y temporadas

La distancia al centroide ayuda a detectar mal ajuste. Las comparaciones por temporada usan porcentajes dentro de fase; la temporada 2026/2027 es parcial en el corpus local.

In [ ]:
# Asignaciones débiles: la marca 'Sin patrón claro' se basa en distancia > umbral del cluster cercano.
df_jugadas["razon_distancia_umbral"] = df_jugadas["distancia_centroide"] / df_jugadas["umbral_distancia_cluster"]
calidad = df_jugadas.groupby("categoria").agg(
    jugadas=("play_id", "size"),
    sin_patron=("cluster_analisis", lambda s: s.eq("Sin patrón claro").sum()),
    distancia_mediana=("distancia_centroide", "median"),
    margen_mediano=("margen_segundo_centroide", "median"),
).reset_index()
calidad["porcentaje_sin_patron"] = 100 * calidad["sin_patron"] / calidad["jugadas"]
display(calidad)

fig, ax = plt.subplots(figsize=(9, 5))
for categoria, datos in df_jugadas.groupby("categoria"):
    ax.hist(datos["razon_distancia_umbral"].dropna().clip(upper=1.4), bins=50,
            density=True, histtype="step", linewidth=2, label=categoria)
ax.axvline(1, linestyle="--", color="black", label="umbral")
ax.set(title="Distancia al centroide relativa al umbral de su cluster", xlabel="Distancia / umbral (cola truncada en 1.4)", ylabel="Densidad")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Composición por familia sugerida, dentro de cada fase y temporada.
temporadas_familias = df_jugadas.groupby(["temporada", "categoria", "familia_sugerida"]).size().rename("jugadas").reset_index()
totales_temporada_fase = temporadas_familias.groupby(["temporada", "categoria"])["jugadas"].transform("sum")
temporadas_familias["porcentaje_temporada_fase"] = 100 * temporadas_familias["jugadas"] / totales_temporada_fase
display(temporadas_familias)
FASE_TEMPORADAS = "juego_abierto"
matriz_temporadas = temporadas_familias.query("categoria == @FASE_TEMPORADAS").pivot(
    index="temporada", columns="familia_sugerida", values="porcentaje_temporada_fase").fillna(0)
ax = matriz_temporadas.plot.bar(stacked=True, figsize=(11, 5), colormap="Set2")
ax.set(title=f"Familias sugeridas por temporada: {FASE_TEMPORADAS.replace('_', ' ')}",
       xlabel="Temporada", ylabel="% de jugadas de la fase")
ax.legend(title="Familia", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Hallazgos de la corrida local (22 de octubre de 2026)

- Se conciliaron **164,660 jugadas** de **1,808 partidos** y **19 equipos**. Balón parado representa 60,852 jugadas (36.96 %) y juego abierto 103,808 (63.04 %).
- En **juego abierto**, la familia sugerida más frecuente es **Progresión** (36,913; 35.56 % de la fase), seguida de **Bajo presión** (28,964; 27.90 %) y **Circulación** (24,762; 23.85 %). La jugada aproximada más frecuente es *Progresión con pases medios y apoyos laterales* (11,337; 10.92 % de la fase).
- En **balón parado**, **Circulación** agrupa 14,524 jugadas (23.87 %), **Reciclaje** 13,110 (21.54 %) y **Progresión** 11,420 (18.77 %). La descripción individual más frecuente es *Reanudación con avance y recuperación; perfil heterogéneo* (10,772; 17.70 %): su carácter mixto exige especial cautela.
- En juego abierto, las jugadas descritas como **Circulación** tienen 8.45 pases completados y 31.28 eventos de media, frente a 5.67 pases y 24.89 eventos de **Bajo presión**. Son diferencias descriptivas de longitud, no de rendimiento.
- **8,243 jugadas (5.0 %)** quedaron en `Sin patrón claro`: 3,046 de balón parado y 5,197 de juego abierto. Esta marca depende de un umbral de distancia; no significa ausencia de estructura futbolística.
- Los nombres de familias y jugadas son **interpretaciones manuales de clusters exploratorios rechazados**. Las comparaciones por equipo deben usar porcentajes dentro de fase y un tamaño mínimo de muestra. La temporada 2026/2027 está incompleta en esta muestra.


## 6. Cómo interpretar los resultados

- `df_jugadas`: una fila por jugada con equipo, temporada, fase, familia y jugada aproximada; conserva los IDs de cluster para auditoría.
- `df_catalogo`: n-gramas principales y ejemplos que sostienen cada descripción.
- `df_estadisticas_jugadas`: frecuencia, pases, eventos, equipos y partidos de cada **jugada aproximada**, con porcentaje dentro de su fase.
- `df_estadisticas_familias`: los mismos resúmenes agrupados por **familia sugerida**.
- `df_estadisticas_equipos`: uso de cada jugada aproximada por equipo y fase. `df_estadisticas_familias_equipos` resume las familias.
- `df_resumen_equipos`: volumen por equipo, partidos y porcentaje sin patrón claro.
- Los equipos tienen distinta cantidad de partidos; usar `porcentaje_fase`, `indice_uso_relativo` y un tamaño mínimo de muestra. La pertenencia al cluster no demuestra por sí sola una jugada táctica específica ni su eficacia.

La última celda permite exportar los dataframes a CSV.

In [ ]:
EXPORTAR_CSV = False
if EXPORTAR_CSV:
    destino = RAIZ / "salidas_analisis_clasificaciones"
    destino.mkdir(exist_ok=True)
    df_catalogo.to_csv(destino / "catalogo_clusters.csv", index=False)
    df_estadisticas_jugadas.to_csv(destino / "estadisticas_jugadas.csv", index=False)
    df_estadisticas_familias.to_csv(destino / "estadisticas_familias.csv", index=False)
    df_estadisticas_equipos.to_csv(destino / "estadisticas_equipos_jugadas.csv", index=False)
    df_estadisticas_familias_equipos.to_csv(destino / "estadisticas_equipos_familias.csv", index=False)
    df_resumen_equipos.to_csv(destino / "resumen_equipos.csv", index=False)
    print(f"CSV guardados en {destino}")